# RealMLP exp_024_val128_lr07 — lr=0.07 @128ep 1-fold 검증 (Kaggle GPU)

lr=0.07 @128ep fold0 vs default-lr 0.944224 / 256ep 0.948021 비교. fold0 단판 (~20분).

**GPU 조건부:** T4(sm_75)면 Kaggle 기본 torch 유지, P100(sm_60)이면 cu121 torch+vision+audio trio 재설치. 마운트 비표준 → glob 자동탐색.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) GPU 종류 감지(nvidia-smi, torch import 前) → 조건부 torch. 그 위에 프로젝트 deps.
GPU = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
if 'P100' in GPU:
    print('P100(sm_60) → cu121 torch trio 재설치 (Kaggle 기본 torch 는 sm_70+ 만)')
    pip('torch==2.5.1','torchvision==0.20.1','torchaudio==2.5.1',
        '--index-url','https://download.pytorch.org/whl/cu121')
else:
    print('T4 등(sm_75+) → Kaggle 기본 torch 유지')
pip('pytabkit','hydra-core','python-dotenv')

In [ ]:
# 3) torch CUDA 실연산 검증 + import 체인 fast-fail (학습 前 무거운 import 확인)
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0))
_x = torch.randn(256, 256, device='cuda'); _v = (_x @ _x).sum().item()
print('CUDA matmul OK')
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_realmlp import run   # hydra/pytabkit/lightning/torchvision import 체인 — 여기서 fail-fast
print('import OK:', config.__file__)

In [ ]:
# 4) 경로 override (cell1 절대경로; working 하위 분리로 이름충돌 방지)
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) repo conf yaml 재사용해 cfg 구성 (Hydra 미사용, OmegaConf 직접)
from omegaconf import OmegaConf
CONF = Path(SRC_ROOT) / "conf"
model_cfg = OmegaConf.load(CONF / "model" / "realmlp.yaml")
model_cfg.params.n_epochs = 128  # 128ep override
model_cfg.params.lr = 0.07       # lr=0.07 override (vs default 0.04)
cfg = OmegaConf.create({
    'exp_id': 'exp_024_val128_lr07',
    'notes': 'lr=0.07 @128ep 1-fold vs default-lr 0.944224 / 256ep 0.948021',
    'use_wandb': False,
    'max_folds': 1,
    'model': model_cfg,
    'features': OmegaConf.load(CONF / 'features' / 'driver_te.yaml'),
    'augment': {'enabled': True, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))

In [ ]:
# 6) 학습 (1-fold 128ep). 예상 ~20분.
import time
t0 = time.time()
result = run(cfg)
print(result, f'\n총 {time.time()-t0:.0f}s')

In [ ]:
# 7) 산출물 확인
oof = pd.read_csv(out / 'oof' / 'exp_024_val128_lr07.csv')
sub = pd.read_csv(out / 'submissions' / 'exp_024_val128_lr07.csv')
print('OOF :', oof.shape, list(oof.columns))
print('SUB :', sub.shape, list(sub.columns))
print(sub['PitNextLap'].describe())